# Chapter 8: LoRA Fine-Tuning
Chapter 7 froze the SmolLM2 backbone entirely. That backbone was pretrained on web text -- it has never seen a robot arm. **LoRA** lets us adapt it with 0.13% of the parameters.

We implement LoRA from scratch (it is two matrices and a scalar), fine-tune our own SmolVLA with it, and then look at the canonical `peft` recipe for OpenVLA-7B.

In [ ]:
!pip install torch torchvision numpy matplotlib transformers pillow

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/08_finetuning

## LoRA in One Cell

A frozen linear layer `W` gets a parallel low-rank branch:

$$h = Wx + \frac{\alpha}{r}\,BAx, \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r}$$

Two details make it work:

- **`B` is zero-initialized**, so `BA = 0` at step 0 and the adapted model starts *exactly* equal to the original. No warm-up shock.
- **The `alpha/r` scaling** means changing `r` does not force you to retune the learning rate.

`W` never receives a gradient. Only `A` and `B` do.

In [ ]:
import torch
from torch import nn
from lora import LoRALinear, LoRAConfig

base = nn.Linear(64, 32)
layer = LoRALinear(base, LoRAConfig(r=8, alpha=16))

x = torch.randn(4, 64)
print(f"scaling alpha/r = {LoRAConfig(r=8, alpha=16).scaling}")
print(f"B is zero-init:   {torch.allclose(layer.lora_B, torch.zeros_like(layer.lora_B))}")
print(f"A shape {tuple(layer.lora_A.shape)}, B shape {tuple(layer.lora_B.shape)}")
print(f"identity at init: {torch.allclose(layer(x), base(x), atol=1e-6)}")

# After perturbing B the branch becomes active.
with torch.no_grad():
    layer.lora_B.normal_(0, 0.02)
print(f"after B != 0:     {torch.allclose(layer(x), base(x), atol=1e-6)}")

## Injecting Adapters into a Backbone

`inject_lora` walks the module tree and wraps every `nn.Linear` whose qualified name contains one of `target_modules`. For a Llama-style backbone -- which SmolLM2 is -- the attention projections are `q_proj`, `k_proj`, `v_proj`, `o_proj`.

Targeting **q and v only** is the LoRA paper's finding: it captures most of the benefit at half the parameters of all-four.

In [ ]:
from lora import inject_lora, mark_only_lora_as_trainable, count_parameters, summarize

class TinyAttention(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

class TinyBackbone(nn.Module):
    def __init__(self, n_layers=6, d=128):
        super().__init__()
        self.layers = nn.ModuleList([TinyAttention(d) for _ in range(n_layers)])

backbone = TinyBackbone()
cfg = LoRAConfig(r=8, alpha=16, target_modules=("q_proj", "v_proj"))

n = inject_lora(backbone, cfg)
mark_only_lora_as_trainable(backbone)

print(f"adapters injected: {n}  (q_proj + v_proj across 6 layers)")
print(summarize(backbone, cfg))

c = count_parameters(backbone)
print(f"\ntotal     {c['total']:>10,}")
print(f"trainable {c['trainable']:>10,}  ({c['trainable']/c['total']*100:.2f}%)")
print(f"frozen    {c['frozen']:>10,}")

## Only the Adapters Move

The claim "the base model is frozen" deserves a check rather than trust. One optimizer step, then compare every parameter before and after.

In [ ]:
before = {n: p.detach().clone() for n, p in backbone.named_parameters()}

opt = torch.optim.SGD([p for p in backbone.parameters() if p.requires_grad], lr=0.1)
h = torch.randn(2, 128)
for layer in backbone.layers:
    h = layer.o_proj(layer.v_proj(layer.q_proj(h)))
h.sum().backward()
opt.step()

moved   = [n for n, p in backbone.named_parameters() if not torch.equal(p, before[n])]
static  = [n for n, p in backbone.named_parameters() if torch.equal(p, before[n])]

print(f"parameters that changed:   {len(moved)}")
print(f"parameters that did not:   {len(static)}")
print(f"all changed are LoRA:      {all('lora_' in n for n in moved)}")
print(f"\nexamples: {moved[:4]}")

# Only lora_B moves on the very first step: with B = 0, the gradient w.r.t. A
# is B^T (...) = 0. A starts moving once B is non-zero.
print(f"\nfirst-step movers are all lora_B: {all(n.endswith('lora_B') for n in moved)}")

## Merging Away the Overhead

At inference you do not want an extra matmul per layer. Because the branch is linear, `W' = W + (alpha/r) BA` folds the adapter into the base weight -- identical outputs, zero runtime cost, and the model becomes a plain `nn.Linear` again.

In [ ]:
from lora import merge_lora

probe = torch.randn(2, 128)
def run(m):
    h = probe
    for layer in m.layers:
        h = layer.o_proj(layer.v_proj(layer.q_proj(h)))
    return h

before_merge = run(backbone)
n_merged = merge_lora(backbone)
after_merge = run(backbone)

print(f"merged adapters: {n_merged}")
print(f"outputs match:   {torch.allclose(before_merge, after_merge, atol=1e-5)}")
print(f"max difference:  {(before_merge - after_merge).abs().max():.2e}")

## Wiring It into Our SmolVLA

`finetune_smolvla.py` injects adapters into the Chapter 7 backbone and drops Ch07's `no_grad` guard so gradients can reach them -- the base weights stay frozen either way. It reuses Ch07's cached vision tokens, so nothing is re-extracted.

`--smoke` exercises the whole wiring with a tiny synthetic model and no downloads.

In [ ]:
!python finetune_smolvla.py --smoke

## Does It Actually Help?

A single fine-tune run against a single baseline proves little -- the gap could be sampling noise. `verify_baseline.py` evaluates both models on **8 seeds with identical noise draws per seed**, so the comparison is paired and the difference is attributable to the adapters.

```bash
uv sync
python finetune_smolvla.py --preset so100 --rank 8     # ~96 s/epoch
python verify_baseline.py --seeds 8 --rank 8
```

In [ ]:
print("SO100 -- 138 train / 20 val episodes, 50 epochs, RTX 4090, paired over 8 seeds\n")
print(f"{'Model':<34}{'Val loss':>18}{'Trainable':>14}")
print("-" * 66)
print(f"{'Ch07 frozen backbone (baseline)':<34}{'0.13474 +/- 0.00123':>18}{'0':>14}")
print(f"{'+ r=8 LoRA on backbone':<34}{'0.12763 +/- 0.00129':>18}{'0.41M (0.13%)':>14}")
print("-" * 66)
print(f"{'Paired improvement':<34}{'0.00711 +/- 0.00014':>18}{'+5.3%':>14}")
print("\nThe paired delta is ~50x its own standard deviation -- real, not noise.")

from IPython.display import Image, display
display(Image("figures/ch08_lora_training_curves.png"))

## The 7B Recipe

Everything above was ours, at ~324M params. `lora_openvla.py` holds the canonical recipe for **OpenVLA-7B**: `peft` + 4-bit NF4 quantization, `r=32`, adapters on *all* linear layers.

It is gated behind dependency detection -- it prints install guidance instead of crashing when `peft` / `bitsandbytes` are missing.

In [ ]:
from lora_openvla import missing_dependencies, OpenVLALoRARecipe

missing = missing_dependencies()
recipe = OpenVLALoRARecipe()
print(recipe.describe())
print(f"\nmissing deps: {missing or 'none'}")
if missing:
    print("  (install with: uv sync --extra openvla)")

print(f"\n{'Approach':<38}{'Trainable':>12}{'% of model':>12}")
print("-" * 62)
for name, tr, pct in [
    ("Ch07 action expert from scratch", "20.8M", "6.4%"),
    ("Ch08 LoRA on backbone (r=8, q/v)", "0.41M", "0.13%"),
    ("OpenVLA-7B full fine-tune", "7B", "100%"),
    ("OpenVLA-7B LoRA (r=32)", "~110M", "~1.5%"),
]:
    print(f"{name:<38}{tr:>12}{pct:>12}")
print("\nFull 7B fine-tuning needs ~140 GB of optimizer state.")
print("LoRA at 4-bit fits on one 24 GB GPU.")

## What We Learned

**LoRA works, and the effect is measurable.** 0.41M trainable params (0.13%) improved validation loss 5.3% over the frozen baseline. The paired protocol matters: with 8 seeds and matched noise draws the delta is ~50x its own std, which is the difference between a result and a hopeful anecdote.

**It overfits fast.** Best validation at epoch ~6, then it drifts back above baseline. 158 episodes is not much data for even 0.41M new parameters. Early stopping is not optional here.

**Not everything worked.** `--train-expert` (r=16, ~21M trainable) diverged around epoch 25 under Chapter 7's constant learning rate with no warmup. It is left in as an exercise rather than quietly dropped -- larger adapters need a schedule the original recipe does not provide.

**Implementing it first makes `peft` obvious.** Once you have written zero-init `B`, the `alpha/r` scaling, and the merge, the OpenVLA recipe reads as a config file rather than magic.

**Next:** Chapter 9 asks the only question that finally matters -- how fast is it, and does it work closed-loop?